# Hands-on Session 2: Running DGAT Protein Inference Workflows

**Time:** 11:05-11:35  
**Lead:** Zeyuan

Goals:
- Prepare transcript and spatial inputs for DGAT-style inference.
- Run the official DGAT workflow or load pretrained DGAT predictions.
- Generate inferred spatial protein landscapes.
- Visualize predicted protein expression patterns.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from dgat_tutorial.data import load_csv_dataset, load_demo_data
from dgat_tutorial.dgat import load_prediction_table, run_demo_dgat_inference
from dgat_tutorial.plotting import plot_spatial_feature

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_dir = repo_root / "data" / "raw"
processed_dir = repo_root / "data" / "processed"
fig_dir = repo_root / "results" / "figures"
processed_dir.mkdir(parents=True, exist_ok=True)
fig_dir.mkdir(parents=True, exist_ok=True)

## Load Tutorial Data

In [ ]:
expected_files = [data_dir / name for name in ["spots.csv", "transcripts.csv", "proteins.csv"]]
dataset = load_csv_dataset(data_dir) if all(path.exists() for path in expected_files) else load_demo_data()

spots = dataset.spots
transcripts = dataset.transcripts
proteins = dataset.proteins

print(transcripts.shape, proteins.shape)

## Prepare DGAT Inputs

For the live tutorial, use the official DGAT repository as the source of model code: https://github.com/osmanbeyoglulab/DGAT

This notebook exports clean aligned tables that can be passed to a DGAT prediction workflow. For committee review and participant backup, it can also load precomputed predictions from `data/raw/dgat_predictions.csv`.

In [ ]:
common_spots = spots.index.intersection(transcripts.index).intersection(proteins.index)
spots = spots.loc[common_spots]
transcripts = transcripts.loc[common_spots]
proteins = proteins.loc[common_spots]

spots.to_csv(processed_dir / "spots_aligned.csv")
transcripts.to_csv(processed_dir / "transcripts_aligned.csv")
proteins.to_csv(processed_dir / "proteins_aligned.csv")

print(f"Prepared {len(common_spots)} aligned spots")

## Run or Load DGAT Predictions

If `data/raw/dgat_predictions.csv` exists, it will be loaded. Otherwise, the notebook uses a small ridge-regression fallback named `run_demo_dgat_inference` so the tutorial remains runnable without the official DGAT environment.

Before the final tutorial release, add the exact official DGAT command you will demonstrate live and keep `data/raw/dgat_predictions.csv` as the backup path.

In [ ]:
prediction_path = data_dir / "dgat_predictions.csv"

if prediction_path.exists():
    predicted_proteins = load_prediction_table(str(prediction_path))
    print("Loaded precomputed DGAT predictions")
else:
    predicted_proteins = run_demo_dgat_inference(transcripts, proteins)
    print("Generated demo DGAT-like predictions")

predicted_proteins.to_csv(processed_dir / "predicted_proteins.csv")
predicted_proteins.head()

## Visualize Inferred Protein Landscapes

In [ ]:
proteins_to_plot = list(predicted_proteins.columns[:4])
fig, axes = plt.subplots(2, 2, figsize=(9, 8))

for ax, protein in zip(axes.ravel(), proteins_to_plot):
    plot_spatial_feature(spots, predicted_proteins[protein], f"Predicted {protein}", ax=ax)

plt.tight_layout()
plt.savefig(fig_dir / "session02_predicted_proteins.png", dpi=160)
plt.show()

## Checkpoint

Participants should now have generated a predicted protein matrix and spatial plots that can be interpreted in Session 3.